In [1]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks
from sklearn.utils import shuffle
from collections import defaultdict
from imblearn.under_sampling import RandomUnderSampler
from sklearn.ensemble import IsolationForest

In [2]:
# Setup all data files as df and sql 
X_train = pd.read_csv("../data/X_train_split.csv")
y_train = pd.read_csv("../data/y_train_split.csv")
X_test = pd.read_csv("../data/X_validation.csv")
y_test = pd.read_csv("../data/y_validation.csv")
X_test_1 = pd.read_csv("../data/X_test_1.csv")
X_test_2 = pd.read_csv("../data/X_test_2.csv")

conn = sqlite3.connect(":memory:")

X_train.to_sql("X_train", conn, index=False, if_exists="replace")
y_train.to_sql("y_train", conn, index=False, if_exists="replace")

8000

In [3]:
# Selecting important features only

important_features = ['17', '46', '48', '71', '86', '87', '88', '90', '99',
                      '111', '141', '172', '218', '231', '234', '249', '263', '270', '283']

In [4]:
X_train = X_train[important_features]
X_test = X_test[important_features]
X_test_1 = X_test_1[important_features]
X_test_2 = X_test_2[important_features]

In [6]:
# Isolation forest to remove potential outliers
iso_models = {}  # save trained iso models per class
X_clean = []
y_clean = []

min_class_count = 50

label_counts = y_train.value_counts()
if isinstance(y_train, pd.DataFrame):
    y_train = y_train.iloc[:, 0]

for label in y_train.unique():
    X_label = X_train[y_train == label]
    y_label = y_train[y_train == label]

    if label_counts[label] >= min_class_count:
        iso = IsolationForest(contamination=0.01, random_state=42, n_jobs=-1)
        iso.fit(X_label)
        iso_models[label] = iso  # save model

        preds = iso.predict(X_label)
        X_clean_label = X_label[preds == 1]
        y_clean_label = y_label[preds == 1]
    else:
        X_clean_label = X_label
        y_clean_label = y_label

    X_clean.append(X_clean_label)
    y_clean.append(y_clean_label)

X_clean = pd.concat(X_clean)
y_clean = pd.concat(y_clean)
X_clean, y_clean = shuffle(X_clean, y_clean, random_state=42)

/Users/james/opt/anaconda3/lib/python3.9/site-packages/sklearn/base.py:450: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(
/Users/james/opt/anaconda3/lib/python3.9/site-packages/sklearn/base.py:450: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(
/Users/james/opt/anaconda3/lib/python3.9/site-packages/sklearn/base.py:450: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(
/Users/james/opt/anaconda3/lib/python3.9/site-packages/sklearn/base.py:450: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(
/Users/james/opt/anaconda3/lib/python3.9/site-packages/sklearn/base.py:450: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(
/Users/james/opt/anaconda3/lib/pyth

In [7]:
X_test_clean = []
y_test_clean = []

label_counts_test = y_test.value_counts()
if isinstance(y_test, pd.DataFrame):
    y_test = y_test.iloc[:, 0]

for label in y_test.unique():
    X_label = X_test[y_test == label]
    y_label = y_test[y_test == label]

    if label in iso_models:
        iso = iso_models[label]
        preds = iso.predict(X_label)
        X_label_clean = X_label[preds == 1]
        y_label_clean = y_label[preds == 1]
    else:
        X_label_clean = X_label
        y_label_clean = y_label

    X_test_clean.append(X_label_clean)
    y_test_clean.append(y_label_clean)

X_test_clean = pd.concat(X_test_clean)
y_test_clean = pd.concat(y_test_clean)

In [8]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clean)
X_test_scaled = scaler.fit_transform(X_test_clean)
X_test_1_scaled = scaler.fit_transform(X_test_1)
X_test_2_scaled = scaler.fit_transform(X_test_2)

In [10]:
X_if_test_1 = pd.DataFrame(X_test_1_scaled)
X_if_test_2 = pd.DataFrame(X_test_2_scaled)

X_if_test_1.to_csv('../data/X_if_test_1.csv', index=False)
X_if_test_2.to_csv('../data/X_if_test_2.csv', index=False)

In [12]:
X_if_test_2

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,-0.428845,0.699202,-0.297349,0.720460,-0.458214,0.252959,-1.073592,0.239369,-0.079964,-0.017399,-0.234234,-0.643499,-0.305835,-0.550653,-0.098064,0.080191,0.988297,0.247401,-0.171489
1,0.291154,-1.103087,-0.239093,0.774412,-0.008825,0.330082,-0.249441,0.257851,0.307747,0.518304,0.067264,-1.023912,-0.118964,0.195498,-0.206579,-0.189922,-0.697354,0.695546,0.659199
2,-0.878156,0.958044,-0.180285,-1.316094,-1.801706,-0.712379,-1.409920,-2.178497,-0.077362,1.113439,-1.257756,0.364894,1.226848,0.876163,-0.701516,-1.976795,0.309227,-1.001156,-0.450437
3,-0.917272,-0.354556,-0.209910,-0.854758,-1.361160,-0.621395,-0.989687,-0.189593,1.320159,-0.135127,-0.183810,0.213537,0.383771,1.279824,-0.880267,-1.181807,0.233099,-0.761692,-1.275350
4,-1.533488,-0.224239,3.206578,-1.835095,-1.817739,-0.653334,-1.974533,-1.540037,-0.896964,2.545909,-1.575439,2.498671,3.184180,0.766847,-1.513575,-1.810540,1.729313,-1.552881,-1.159954
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2015,-0.061155,1.587863,0.372946,0.880238,1.229163,1.620055,-0.644803,1.677258,-1.117322,-1.419850,-0.158392,-0.237229,-1.834591,0.736641,1.202646,1.105339,0.982334,-0.500929,0.531072
2016,0.688008,-0.219304,-0.168862,-0.483883,-0.302901,-0.003944,0.197405,-0.142553,-0.670204,0.921513,0.563546,-0.245121,0.425782,-0.422254,-0.296205,-0.551633,0.854756,0.280653,-0.529628
2017,0.473440,0.482826,-0.520459,-0.449346,-0.077344,-0.397875,0.298500,-0.451449,0.434540,-0.414931,0.160072,-0.216646,-0.496296,1.200034,-0.532227,0.018527,-0.067770,-0.421657,0.280898
2018,0.031773,-0.399001,0.367168,1.346603,0.167557,0.906222,-0.625500,1.043113,-0.401995,-0.979871,1.253799,0.027436,-0.279381,-0.653638,-1.597258,0.631603,1.803758,0.100085,-1.079909


In [28]:
def SMOTE_TOMEK(X_train, y_train):
    
    print(f"Initial label distribution:\n{y_train.value_counts()}")
    
    # Smote low count labels first
    
    smote_low = SMOTE(
        sampling_strategy={1: 50, 2: 50, 16: 50, 22:50},  
        k_neighbors=3,
        random_state=42
    )

    X_smote_low, y_smote_low = smote_low.fit_resample(X_train,y_train)
    
    print(f"Label distribution after low count smote step:\n{y_smote_low.value_counts()}")
    
    # Smote remaining labels
    
    smote = SMOTE(
        sampling_strategy={
            12: 500,   # 457 -> resample to 500
            24: 500,   # 385 -> resample to 500
            17: 500,   # 354 -> resample to 500
            26: 500,   # 278 -> resample to 500
            21: 500,   # 269 -> resample to 500
            14: 500,   # 264 -> resample to 500
            4: 500,    # 238 -> resample to 500
            25: 500,   # 184 -> resample to 500
            19: 500,   # 177 -> resample to 500
            20: 500,   # 153 -> resample to 500
            27: 200,   # 107 -> resample to 200
            7: 200,    # 103 -> resample to 200
            11: 200,   # 78 -> resample to 200
            3: 200,    # 65 -> resample to 200
            13: 200,   # 62 -> resample to 200
            18: 200,   # 60 -> resample to 200
            23: 200,   # 41 -> resample to 200
            15: 200,   # 26 -> resample to 200
            9: 200,    # 25 -> resample to 200
            0: 200     # 18 -> resample to 200
        },  
        k_neighbors=5,
        random_state=42
    )
    
    X_smote, y_smote = smote.fit_resample(X_smote_low,y_smote_low)
    
    print(f"Label distribution after regular smote step:\n{y_smote.value_counts()}")
    
    # Undersample using TomekLinks
    
    tomek = TomekLinks(sampling_strategy='not minority') 
    X_tomek, y_tomek = tomek.fit_resample(X_smote, y_smote)
    
    print(f"Label distribution after removing tomek links:\n{y_tomek.value_counts()}")
    
    return X_tomek, y_tomek

In [29]:
def SMOTE_TOMEK_UNDER(X_train, y_train):
    
    print(f"Initial label distribution:\n{y_train.value_counts()}")
    
    # Smote low count labels first
    
    smote_low = SMOTE(
        sampling_strategy={1: 50, 2: 50, 16: 50, 22:50},  
        k_neighbors=3,
        random_state=42
    )

    X_smote_low, y_smote_low = smote_low.fit_resample(X_train,y_train)
    
    print(f"Label distribution after low count smote step:\n{y_smote_low.value_counts()}")
    
    # Smote remaining labels
    
    smote = SMOTE(
        sampling_strategy={
            12: 500,   # 457 -> resample to 500
            24: 500,   # 385 -> resample to 500
            17: 500,   # 354 -> resample to 500
            26: 500,   # 278 -> resample to 500
            21: 500,   # 269 -> resample to 500
            14: 500,   # 264 -> resample to 500
            4: 500,    # 238 -> resample to 500
            25: 500,   # 184 -> resample to 500
            19: 500,   # 177 -> resample to 500
            20: 500,   # 153 -> resample to 500
            27: 200,   # 107 -> resample to 200
            7: 200,    # 103 -> resample to 200
            11: 200,   # 78 -> resample to 200
            3: 200,    # 65 -> resample to 200
            13: 200,   # 62 -> resample to 200
            18: 200,   # 60 -> resample to 200
            23: 200,   # 41 -> resample to 200
            15: 200,   # 26 -> resample to 200
            9: 200,    # 25 -> resample to 200
            0: 200     # 18 -> resample to 200
        },  
        k_neighbors=5,
        random_state=42
    )
    
    X_smote, y_smote = smote.fit_resample(X_smote_low,y_smote_low)
    
    print(f"Label distribution after regular smote step:\n{y_smote.value_counts()}")
    
    # Undersample using TomekLinks
    
    tomek = TomekLinks(sampling_strategy='not minority') 
    X_tomek, y_tomek = tomek.fit_resample(X_smote, y_smote)
    
    print(f"Label distribution after removing tomek links:\n{y_tomek.value_counts()}")
    
    rus = RandomUnderSampler(
        sampling_strategy={5:1000},
        random_state=42
    )
    X_under, y_under = rus.fit_resample(X_tomek, y_tomek)
    
    print(f"Label distribution after undersampling class 5:\n{y_under.value_counts()}")
    
    return X_under, y_under

In [30]:
# Set all labels to roughly the same number of points of 500
def balance(X_train, y_train):
    
    print(f"Initial label distribution:\n{y_train.value_counts()}")
    
    # Smote low count labels first
    
    smote_low = SMOTE(
        sampling_strategy={1: 1000, 2: 1000, 16: 1000, 22:1000},  
        k_neighbors=3,
        random_state=42
    )

    X_smote_low, y_smote_low = smote_low.fit_resample(X_train,y_train)
    
    print(f"Label distribution after low count smote step:\n{y_smote_low.value_counts()}")
    
    # Smote remaining labels
    
    smote = SMOTE(
        sampling_strategy={
            10: 1000,
            6: 1000,
            8: 1000,
            12: 1000,   # 457 -> resample to 500
            24: 1000,   # 385 -> resample to 500
            17: 1000,   # 354 -> resample to 500
            26: 1000,   # 278 -> resample to 500
            21: 1000,   # 269 -> resample to 500
            14: 1000,   # 264 -> resample to 500
            4: 1000,    # 238 -> resample to 500
            25: 1000,   # 184 -> resample to 500
            19: 1000,   # 177 -> resample to 500
            20: 1000,   # 153 -> resample to 500
            27: 1000,   # 107 -> resample to 200
            7: 1000,    # 103 -> resample to 200
            11: 1000,   # 78 -> resample to 200
            3: 1000,    # 65 -> resample to 200
            13: 1000,   # 62 -> resample to 200
            18: 1000,   # 60 -> resample to 200
            23: 1000,   # 41 -> resample to 200
            15: 1000,   # 26 -> resample to 200
            9: 1000,    # 25 -> resample to 200
            0: 1000     # 18 -> resample to 200
        },  
        k_neighbors=5,
        random_state=42
    )
    
    X_smote, y_smote = smote.fit_resample(X_smote_low,y_smote_low)
    
    print(f"Label distribution after regular smote step:\n{y_smote.value_counts()}")
    
    # Undersample using TomekLinks
    
    tomek = TomekLinks(sampling_strategy='not minority') 
    X_tomek, y_tomek = tomek.fit_resample(X_smote, y_smote)
    
    print(f"Label distribution after removing tomek links:\n{y_tomek.value_counts()}")
    
    rus = RandomUnderSampler(
        sampling_strategy={5:1000},
        random_state=42
    )
    X_under, y_under = rus.fit_resample(X_tomek, y_tomek)
    
    print(f"Label distribution after undersampling class 5:\n{y_under.value_counts()}")
    
    return X_under, y_under

In [32]:
X_smotetomek, y_smotetomek = SMOTE_TOMEK(X_scaled, y_clean)

Initial label distribution:
5     3547
10     856
6      437
8      408
12     362
24     304
17     280
26     219
21     212
14     208
4      188
25     145
19     140
20     120
27      85
7       81
11      61
3       51
13      49
18      48
23      33
15      21
9       20
0       14
2        6
1        6
22       6
16       5
Name: label, dtype: int64
Label distribution after low count smote step:
5     3547
10     856
6      437
8      408
12     362
24     304
17     280
26     219
21     212
14     208
4      188
25     145
19     140
20     120
27      85
7       81
11      61
3       51
2       50
1       50
16      50
22      50
13      49
18      48
23      33
15      21
9       20
0       14
Name: label, dtype: int64
Label distribution after regular smote step:
5     3547
10     856
21     500
19     500
24     500
14     500
4      500
12     500
25     500
20     500
26     500
17     500
6      437
8      408
11     200
0      200
13     200
7      200
23     200
3  

In [33]:
X_smotetomekunder, y_smotetomekunder = SMOTE_TOMEK_UNDER(X_scaled, y_clean)

Initial label distribution:
5     3547
10     856
6      437
8      408
12     362
24     304
17     280
26     219
21     212
14     208
4      188
25     145
19     140
20     120
27      85
7       81
11      61
3       51
13      49
18      48
23      33
15      21
9       20
0       14
2        6
1        6
22       6
16       5
Name: label, dtype: int64
Label distribution after low count smote step:
5     3547
10     856
6      437
8      408
12     362
24     304
17     280
26     219
21     212
14     208
4      188
25     145
19     140
20     120
27      85
7       81
11      61
3       51
2       50
1       50
16      50
22      50
13      49
18      48
23      33
15      21
9       20
0       14
Name: label, dtype: int64
Label distribution after regular smote step:
5     3547
10     856
21     500
19     500
24     500
14     500
4      500
12     500
25     500
20     500
26     500
17     500
6      437
8      408
11     200
0      200
13     200
7      200
23     200
3  

In [34]:
X_balance, y_balance = balance(X_scaled, y_clean)

Initial label distribution:
5     3547
10     856
6      437
8      408
12     362
24     304
17     280
26     219
21     212
14     208
4      188
25     145
19     140
20     120
27      85
7       81
11      61
3       51
13      49
18      48
23      33
15      21
9       20
0       14
2        6
1        6
22       6
16       5
Name: label, dtype: int64
Label distribution after low count smote step:
5     3547
2     1000
16    1000
1     1000
22    1000
10     856
6      437
8      408
12     362
24     304
17     280
26     219
21     212
14     208
4      188
25     145
19     140
20     120
27      85
7       81
11      61
3       51
13      49
18      48
23      33
15      21
9       20
0       14
Name: label, dtype: int64
Label distribution after regular smote step:
5     3547
19    1000
16    1000
1     1000
2     1000
0     1000
13    1000
7     1000
23    1000
11    1000
3     1000
27    1000
15    1000
9     1000
25    1000
24    1000
14    1000
4     1000
6     1000
12 

In [35]:
# Export new synthetic datasets

# Convert X_final (features) to DataFrame
X_test_if = pd.DataFrame(X_test_clean)
y_test_if = pd.DataFrame(y_test_clean, columns=['label'])
X_if = pd.DataFrame(X_scaled)
y_if = pd.DataFrame(y_clean, columns=['label'])

# Export datasets to CSV
X_test_if.to_csv('../data/X_if_validation.csv', index=False)
y_test_if.to_csv('../data/y_if_validation.csv', index=False)
X_if.to_csv('../data/X_if_train.csv', index=False)
y_if.to_csv('../data/y_if_train.csv', index=False)

# Convert to DataFrame
X_balance = pd.DataFrame(X_balance)
X_smotetomek = pd.DataFrame(X_smotetomek)
X_smotetomekunder = pd.DataFrame(X_smotetomekunder)

# Convert labels to DataFrame (with a column name)
y_balance = pd.DataFrame(y_balance, columns=['label'])
y_smotetomek = pd.DataFrame(y_smotetomek, columns=['label'])
y_smotetomekunder = pd.DataFrame(y_smotetomekunder, columns=['label'])

# Export X-df to CSV
X_balance.to_csv('../data/X_if_balance.csv', index=False)
X_smotetomek.to_csv('../data/X_if_SMOTETOMEK.csv', index=False)
X_smotetomekunder.to_csv('../data/X_if_SMOTETOMEKUNDER.csv', index=False)

y_balance.to_csv('../data/y_if_balance.csv', index=False)
y_smotetomek.to_csv('../data/y_if_SMOTETOMEK.csv', index=False)
y_smotetomekunder.to_csv('../data/y_if_SMOTETOMEKUNDER.csv', index=False)


In [54]:
X_balance

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,1.756607,-0.181274,-1.252341,0.399347,1.012100,2.172247,1.021140,0.374845,-0.950206,-0.579274,1.327273,-0.495730,-1.077435,-1.515487,0.647688,0.453148,-1.056428,0.475359,0.905797
1,0.786229,-0.377232,-1.190856,-0.390694,1.523140,1.157173,0.511067,0.263494,-0.823890,-0.111156,0.454854,-1.660330,-1.153815,-0.956043,0.797707,0.979710,-0.623923,0.522503,0.955474
2,-0.440278,-1.372375,1.051177,1.844846,2.682958,2.332386,-0.043054,0.542358,-1.588750,-1.098812,1.288683,0.746864,-0.760563,-2.093635,0.638579,2.054904,-0.539163,-1.500458,-0.629866
3,1.229077,0.158866,-1.050620,0.706888,0.731269,0.079497,1.167433,0.893365,1.406834,-1.315109,0.719527,-0.961149,-0.546186,-0.880040,1.418464,0.615087,-0.409479,0.981386,0.104424
4,0.729050,-0.435114,-0.586515,1.476237,0.401854,0.974810,1.193248,0.812740,0.603105,-1.096362,0.233610,-0.099629,-0.743379,-0.240290,1.246426,1.285555,0.043869,0.357252,1.594451
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27989,0.786691,0.245147,-1.043658,0.832927,0.627635,-0.204895,0.338048,0.659197,1.457883,-1.131949,0.377265,-1.798336,0.322476,-0.084581,0.640151,0.002670,-0.354794,0.549739,0.818195
27990,0.822591,-1.147014,-1.362145,1.094833,-0.524293,0.335526,0.569491,0.440384,2.284136,-1.083381,0.957889,-0.023844,-1.217520,0.554432,-0.482455,0.399318,-0.713576,0.318155,0.588048
27991,-0.202875,-1.320334,0.088858,0.877367,0.370706,-0.556673,0.395651,0.121648,1.395192,-1.108438,1.197157,0.098095,-1.023552,0.484445,0.123545,0.936063,-0.054349,-0.024093,-0.520739
27992,0.094858,-0.659847,-0.231121,0.880431,-0.436735,0.365083,0.110609,0.184558,1.495981,-0.367336,0.108529,0.733140,-1.161219,0.218497,-0.225978,0.548630,-0.207507,0.005368,0.107942
